In [5]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("NYC_Taxi_BigData_Processing") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("Spark initialized successfully!")
print(f"Spark version: {spark.version}")

from google.colab import drive
drive.mount('/content/drive')

RAW_DATA_PATH = "/content/drive/MyDrive/dataset_nyc/*.parquet"
OUTPUT_PATH = "/content/drive/MyDrive/ProcessedData"

print("Reading raw parquet files...")
df_raw = spark.read.parquet(RAW_DATA_PATH)

print(f"Total records: {df_raw.count():,}")
print("Schema:")
df_raw.printSchema()

print("\nCleaning data...")

df_clean = df_raw \
    .filter(col("trip_distance") > 0) \
    .filter(col("total_amount") > 0) \
    .filter(col("passenger_count") > 0) \
    .filter(col("passenger_count") <= 6) \
    .filter(col("trip_distance") <= 100) \
    .filter(col("total_amount") <= 500) \
    .filter(col("fare_amount") > 0) \
    .dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"])

print(f"Records after cleaning: {df_clean.count():,}")

print("\nFeature engineering...")

df_clean = df_clean \
    .withColumn("pickup_datetime", to_timestamp("tpep_pickup_datetime")) \
    .withColumn("dropoff_datetime", to_timestamp("tpep_dropoff_datetime"))

df_clean = df_clean.withColumn(
    "trip_duration_mins",
    (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60
)

df_clean = df_clean.filter(
    (col("trip_duration_mins") >= 1) & (col("trip_duration_mins") <= 180)
)

df_clean = df_clean \
    .withColumn("pickup_hour", hour("pickup_datetime")) \
    .withColumn("pickup_dayofweek", dayofweek("pickup_datetime")) \
    .withColumn("pickup_date", to_date("pickup_datetime")) \
    .withColumn("year", year("pickup_datetime")) \
    .withColumn("month", month("pickup_datetime"))

df_dashboard = df_clean.select(
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "total_amount",
    "fare_amount",
    "trip_duration_mins",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_date",
    "year",
    "month",
    "passenger_count"
)

print("Feature engineering completed!")

print("\nSaving clean data for dashboard...")

df_dashboard.write \
    .partitionBy("year", "month") \
    .mode("overwrite") \
    .parquet(f"{OUTPUT_PATH}/clean_taxi_data.parquet")

print(f"Saved to: {OUTPUT_PATH}/clean_taxi_data.parquet")

print("\nCreating streaming sample (latest 100K records)...")

df_streaming = df_dashboard \
    .orderBy(desc("pickup_datetime")) \
    .limit(100000)

df_streaming.write \
    .mode("overwrite") \
    .parquet(f"{OUTPUT_PATH}/streaming_sample.parquet")

print(f"Streaming sample saved: {OUTPUT_PATH}/streaming_sample.parquet")

print("\nAggregating time series data for AI model...")

df_timeseries = df_clean.groupBy(
    "pickup_date",
    "pickup_hour",
    "PULocationID"
).agg(
    count("*").alias("demand_count"),
    avg("trip_distance").alias("avg_distance"),
    avg("total_amount").alias("avg_fare"),
    avg("trip_duration_mins").alias("avg_duration"),
    sum("total_amount").alias("total_revenue")
).orderBy("pickup_date", "pickup_hour", "PULocationID")

df_timeseries = df_timeseries.withColumn(
    "timestamp",
    to_timestamp(concat(col("pickup_date"), lit(" "), col("pickup_hour"), lit(":00:00")))
)

df_timeseries = df_timeseries \
    .withColumn("dayofweek", dayofweek("timestamp")) \
    .withColumn("is_weekend", when(col("dayofweek").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("month", month("timestamp")) \
    .withColumn("year", year("timestamp"))

df_model = df_timeseries.select(
    "timestamp",
    "PULocationID",
    "pickup_hour",
    "dayofweek",
    "is_weekend",
    "month",
    "year",
    "demand_count",
    "avg_distance",
    "avg_fare",
    "avg_duration",
    "total_revenue"
).orderBy("PULocationID", "timestamp")

print(f"Time series records: {df_model.count():,}")

df_model.write \
    .partitionBy("PULocationID") \
    .mode("overwrite") \
    .parquet(f"{OUTPUT_PATH}/timeseries_taxi.parquet")

print(f"Saved to: {OUTPUT_PATH}/timeseries_taxi.parquet")

print("\nCreating zone mapping file...")

TAXI_ZONES = {
    1: "Newark Airport", 4: "Alphabet City", 13: "Battery Park", 24: "Bloomingdale",
    43: "Central Park", 48: "Clinton East", 68: "East Chelsea", 79: "East Village",
    87: "Financial District South", 88: "Financial District North", 100: "Garment District",
    107: "Gramercy", 113: "Greenwich Village North", 125: "Harlem North", 137: "Kips Bay",
    140: "Lenox Hill East", 142: "Lincoln Square East", 143: "Lincoln Square West",
    148: "Lower East Side", 151: "Manhattanville", 158: "Meatpacking/West Village West",
    161: "Midtown Center", 162: "Midtown East", 163: "Midtown North", 164: "Midtown South",
    166: "Morningside Heights", 170: "Murray Hill", 186: "Penn Station/Madison Sq West",
    202: "Seaport", 209: "SoHo", 211: "South Williamsburg", 224: "Stuy Town/Peter Cooper Village",
    229: "Sutton Place/Turtle Bay North", 230: "Times Sq/Theatre District",
    231: "TriBeCa/Civic Center", 232: "Two Bridges/Seward Park", 233: "UN/Turtle Bay South",
    234: "Union Sq", 236: "Upper East Side North", 237: "Upper East Side South",
    238: "Upper West Side North", 239: "Upper West Side South", 243: "Washington Heights North",
    244: "Washington Heights South", 246: "West Chelsea/Hudson Yards", 249: "West Village"
}

zone_data = [(zone_id, zone_name) for zone_id, zone_name in TAXI_ZONES.items()]
df_zones = spark.createDataFrame(zone_data, ["zone_id", "zone_name"])

df_zones.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/zone_mapping.parquet")
print(f"Zone mapping saved: {OUTPUT_PATH}/zone_mapping.parquet")

print("\n" + "="*80)
print("PROCESSING SUMMARY")
print("="*80)

print(f"\nOutput Files:")
print(f"   1. clean_taxi_data.parquet (Dashboard input)")
print(f"   2. timeseries_taxi.parquet (AI Model input)")
print(f"   3. streaming_sample.parquet (Real-time simulation)")
print(f"   4. zone_mapping.parquet (Zone names)")

print(f"\nData Statistics:")
df_clean_sample = spark.read.parquet(f"{OUTPUT_PATH}/clean_taxi_data.parquet")
print(f"   - Total clean records: {df_clean_sample.count():,}")

df_ts_sample = spark.read.parquet(f"{OUTPUT_PATH}/timeseries_taxi.parquet")
print(f"   - Time series records: {df_ts_sample.count():,}")
print(f"   - Unique zones: {df_ts_sample.select('PULocationID').distinct().count()}")

date_range = df_ts_sample.agg(
    min("timestamp").alias("min_date"),
    max("timestamp").alias("max_date")
).collect()[0]

print(f"   - Date range: {date_range['min_date']} to {date_range['max_date']}")

print("\nALL PROCESSING COMPLETED SUCCESSFULLY!")
print("="*80)

spark.stop()

✅ Spark initialized successfully!
Spark version: 4.0.2
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Reading raw parquet files...
Total records: 157,973,451
Schema:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: d